# Physics-penalty sweep: compare the three runs

Run this AFTER `03a` / `03b` / `03c` have each finished and saved to Drive.
It pulls the three models + evals back from Drive and compares them. No GPU needed.

Reads: `fno_grains_{physics,tuned,nophysics}.pt` and `fno_grains_eval_{...}.zip` from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torch pandas

### Pull the three runs back from Drive

In [ ]:
import os
for tag in ['physics', 'tuned', 'nophysics']:
    !cp /content/drive/MyDrive/fno_grains_eval_{tag}.zip .
    !unzip -o -q fno_grains_eval_{tag}.zip            # restores runs/ablation_{tag}/eval/...
    !mkdir -p runs/ablation_{tag}
    !cp /content/drive/MyDrive/fno_grains_{tag}.pt runs/ablation_{tag}/best.pt
print('pulled physics / tuned / nophysics from Drive')

### Compare
Success for `tuned` (0.02): beats **both** `physics` (0.1) and `nophysics` (0.0) overall,
lands close to `nophysics` on hex/point AND close to `physics` on multi. The `best_epoch`
column also settles whether 0.1 was stalling training early.

In [ ]:
import pandas as pd, torch
WEIGHTS = {'physics': 0.1, 'tuned': 0.02, 'nophysics': 0.0}
rows = []
for tag in ['physics', 'tuned', 'nophysics']:
    df = pd.read_csv(f'runs/ablation_{tag}/eval/per_trajectory.csv')
    ck = torch.load(f'runs/ablation_{tag}/best.pt', map_location='cpu', weights_only=False)
    rows.append({
        'run': tag,
        'physics_weight': WEIGHTS[tag],
        'best_epoch': ck.get('epoch'),
        'best_val_loss': ck.get('val_loss'),
        'mean_rollout_mse': round(float(df['rollout_mse'].mean()), 6),
        'mean_spectral_mse': round(float(df['spectral_mse'].mean()), 6),
        'phase_offset_trajs': int(df['phase_offset'].sum()),
        'n_traj': len(df),
    })
display(pd.DataFrame(rows))

In [ ]:
# Per-seed-type rollout MSE, three runs side by side
import pandas as pd
frames = []
for tag in ['physics', 'tuned', 'nophysics']:
    df = pd.read_csv(f'runs/ablation_{tag}/eval/per_trajectory.csv')
    frames.append(df.groupby('seed_type')['rollout_mse'].mean().rename(tag))
display(pd.concat(frames, axis=1))